# 10 · 毕业项目：前缀感知路由

这是全课收尾，也是**可以直接写进简历的那个项目**。

问题：第 09 章证明了 prompt 结构影响单副本内的缓存命中。但线上一定是多副本的——请求被负载均衡器轮流传给各个副本，**同一个前缀在不同副本上各算一遍**，命中率被副本数稀释。

解法：在负载均衡层做**前缀感知路由**，把前缀相同的请求尽量送到同一个副本上。

这一章用真实计算搭出这个系统，并对比四种路由策略。

In [ ]:
# ===== 引导单元：环境检查 + 测量工具 + MiniGPT（每章自带，直接运行）=====
# 说明：本单元在每个 notebook 里都有一份完整副本，目的是让任何一个 notebook
#       都能在 Colab 里零配置独立运行。想改模型结构，请改 tools/build_notebooks.py
#       里的 SETUP_CODE，然后重跑编译脚本。
#
# 架构对齐：下面这套推理核心刻意模仿了 vLLM V1 的模块划分与命名，
#   详见 docs/vllm-mapping.md 的对照表。
#       EngineCore.step()           ←→ vllm/v1/engine/core.py
#         ├─ Scheduler.schedule()   ←→ vllm/v1/core/sched/scheduler.py
#         ├─ ModelRunner.execute_model() ←→ vllm/v1/worker/gpu_model_runner.py
#         └─ Scheduler.update_from_output()
import math
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# MiniGPT 只有 2700 万参数，用 float16 跑在 GPU 上；CPU 上 float16 很慢，用 float32
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32


def sync():
    """GPU 是异步执行的，计时前必须同步，否则测到的是下发时间不是执行时间。"""
    if DEVICE == "cuda":
        torch.cuda.synchronize()


def bench(fn, warmup=3, iters=10):
    """返回单次调用的平均耗时（毫秒）。warmup 用来排除首次 kernel 编译等开销。"""
    for _ in range(warmup):
        fn()
    sync()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    sync()
    return (time.perf_counter() - t0) / iters * 1000.0


def peak_mem_mb():
    """当前 CUDA 峰值显存占用（MB）。"""
    if DEVICE != "cuda":
        return 0.0
    return torch.cuda.max_memory_allocated() / 1024 ** 2


def reset_peak():
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()


class Config:
    def __init__(self, vocab_size=50257, block_size=1024, n_layer=4, n_head=6, n_embd=384):
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_embd = n_embd
        self.head_dim = n_embd // n_head


class CausalSelfAttention(nn.Module):
    """因果自注意力，支持 KV cache。

    past_kv 传入历史的 (k, v)，本步只为新 token 计算 Q/K/V，然后拼在历史后面。
    返回 (输出, 更新后的 (k, v))，其中 k/v 的 shape 是 (B, n_head, 总长度, head_dim)。
    """

    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.head_dim = cfg.head_dim
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)

    def forward(self, x, past_kv=None, attn_mask=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)

        S = k.size(2)  # 总长度 = 历史 + 本步新增
        if attn_mask is None:
            # 默认因果掩码：本步第 i 个 query 的绝对位置是 S-T+i，只能看见 <= 它的 key
            mask = torch.ones(T, S, device=x.device).tril(diagonal=S - T).bool()
        else:
            # 外部传入的掩码，用于一个 batch 里混合不同进度的序列（第 04、06 章）
            mask = attn_mask
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y), (k, v)


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc = nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=False)

    def forward(self, x):
        return self.proj(F.gelu(self.fc(x)))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg)

    def forward(self, x, past_kv=None, attn_mask=None):
        h, present = self.attn(self.ln_1(x), past_kv, attn_mask)
        x = x + h
        x = x + self.mlp(self.ln_2(x))
        return x, present


class MiniGPT(nn.Module):
    """极简 GPT，结构与 Llama 同源：pre-norm + 因果注意力 + 4 倍扩张 MLP + 权重共享。

    与 Llama 的两处差异：
      - 用可学习位置编码代替 RoPE（简化实现，不影响调度实验的结论）
      - 没有 GQA（本仓库是 MHA，第 03 章会手工比较两者的 KV cache 大小）
    """

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.wpe = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight  # 权重共享，省一份 embedding 参数

        def init(m):
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)

        self.apply(init)

    def forward(self, idx, past_kvs=None, pos_offset=0, attn_mask=None):
        """idx: (B, T) 的 token id。

        past_kvs: 长度等于层数的列表，每项是 (k, v)；None 表示从零开始（prefill）。
        pos_offset: 本次输入的第一个 token 的绝对位置。传 int 表示整个 batch 用同一个
                    偏移；传 shape (B,) 的张量表示每条序列各用各的偏移——当 batch 里
                    混合了不同进度的请求时必须这样传。
        attn_mask: 可选的自定义注意力掩码，用于屏蔽填充位。
        """
        B, T = idx.shape
        if torch.is_tensor(pos_offset):
            pos = pos_offset.view(B, 1) + torch.arange(T, device=idx.device)[None, :]
        else:
            pos = torch.arange(pos_offset, pos_offset + T, device=idx.device)[None, :].expand(B, T)
        x = self.wte(idx) + self.wpe(pos)

        presents = []
        for i, blk in enumerate(self.blocks):
            past = None if past_kvs is None else past_kvs[i]
            x, present = blk(x, past, attn_mask)
            presents.append(present)
        return self.lm_head(self.ln_f(x)), presents

    @property
    def n_params(self):
        return sum(p.numel() for p in self.parameters())


def build_model(seed=0, device=DEVICE, dtype=DTYPE, **kw):
    torch.manual_seed(seed)
    cfg = Config(**kw)
    model = MiniGPT(cfg).to(device=device, dtype=dtype)
    return model.eval()


@torch.no_grad()
def generate_naive(model, idx, max_new_tokens):
    """不用 KV cache：每一步都把完整序列重新算一遍（O(n^2) 重算）。"""
    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -model.cfg.block_size:])
        idx = torch.cat([idx, logits[:, -1].argmax(-1, keepdim=True)], dim=1)
    return idx


@torch.no_grad()
def generate_cached(model, idx, max_new_tokens):
    """用 KV cache：prompt 只 prefill 一次，之后每步只喂 1 个 token。"""
    logits, past = model(idx)
    nxt = logits[:, -1].argmax(-1, keepdim=True)
    out = [nxt]
    pos = idx.size(1)
    for _ in range(max_new_tokens - 1):
        logits, past = model(nxt, past_kvs=past, pos_offset=pos)
        pos += 1
        nxt = logits[:, -1].argmax(-1, keepdim=True)
        out.append(nxt)
    return torch.cat([idx] + out, dim=1)


def kv_bytes(n_layer, n_kv_head, head_dim, seq_len, batch=1, dtype_bytes=2):
    """KV cache 字节数。注意是 2（K 和 V 各一份）。"""
    return 2 * n_layer * n_kv_head * head_dim * seq_len * batch * dtype_bytes


# ========== 以下是模仿 vLLM V1 架构的推理核心 ==========


class Request:
    """对应 vllm/v1/request.py 的 Request。

    num_computed_tokens 是 vLLM 里最核心的一个字段：它记录这条请求已经有
    多少 token 的 KV 被算过。prefill、chunked prefill、前缀缓存命中——
    三种看起来完全不同的场景，在 vLLM 里都只是「把 num_computed_tokens 往前推」。
    理解这一点，chunked prefill 就不再是独立机制，而是这个字段的自然结果。
    """

    def __init__(self, request_id, prompt_token_ids, max_tokens):
        self.request_id = request_id
        self.prompt_token_ids = list(prompt_token_ids)
        self.max_tokens = max_tokens
        self.output_token_ids = []
        self.num_computed_tokens = 0
        self.status = "waiting"      # waiting / running / finished
        # 本仓库简化：直接把 KV 张量挂在请求上。
        # 真实 vLLM 不这么做——请求只持有 block_table，物理 block 由 KVCacheManager 管（第 05 章）。
        self.past = None

    @property
    def num_prompt_tokens(self):
        return len(self.prompt_token_ids)

    def all_token_ids(self):
        return self.prompt_token_ids + self.output_token_ids

    def num_tokens_to_schedule(self):
        """还欠多少 token 没算：prefill 阶段是剩余 prompt 长度，decode 阶段是 1。"""
        if self.num_computed_tokens < self.num_prompt_tokens:
            return self.num_prompt_tokens - self.num_computed_tokens
        return 1

    @property
    def is_finished(self):
        return len(self.output_token_ids) >= self.max_tokens

    def __repr__(self):
        return (f"Request({self.request_id}, computed={self.num_computed_tokens}"
                f"/{self.num_prompt_tokens}, out={len(self.output_token_ids)}"
                f"/{self.max_tokens}, {self.status})")


class SchedulerOutput:
    """对应 vllm/v1/core/sched/output.py 的 SchedulerOutput。

    调度与执行之间唯一的接口。真实 vLLM 里这个结构还包含 block 分配结果、
    抢占列表等字段，这里只保留最必要的两个。
    """

    def __init__(self, scheduled_reqs, num_scheduled_tokens):
        self.scheduled_reqs = scheduled_reqs
        self.num_scheduled_tokens = num_scheduled_tokens   # {request_id: n}

    def __len__(self):
        return len(self.scheduled_reqs)


class Scheduler:
    """对应 vllm/v1/core/sched/scheduler.py 的 Scheduler。

    职责边界是这个架构里最值得学的一点：Scheduler 只决定
    「这一轮跑哪些请求、各自跑几个 token」，它既不碰显存也不碰模型。

        显存分配 → KVCacheManager（第 05 章）
        真正计算 → ModelRunner

    三个模块分离，才能各自独立替换实现。面试被问「说说 vLLM 的架构」时，
    先把这个职责划分讲清楚，比背模块名有用得多。
    """

    def __init__(self, max_num_seqs=8, max_num_batched_tokens=2048):
        self.waiting = []
        self.running = []
        self.finished = []
        self.max_num_seqs = max_num_seqs
        # 这个预算就是 chunked prefill 的开关：调小它，长 prompt 自然被切成多轮（第 06 章）
        self.max_num_batched_tokens = max_num_batched_tokens
        self.step_id = 0

    def add_request(self, req):
        self.waiting.append(req)

    def has_unfinished(self):
        return bool(self.waiting or self.running)

    def schedule(self):
        scheduled, num_tokens = [], {}
        budget = self.max_num_batched_tokens

        # 第一优先：正在跑的请求。已进 decode 的排 1 个 token；
        # 还在做 chunked prefill 的按剩余量排，但受 budget 限制。
        for req in list(self.running):
            if budget <= 0 or len(scheduled) >= self.max_num_seqs:
                break
            n = min(req.num_tokens_to_schedule(), budget)
            scheduled.append(req)
            num_tokens[req.request_id] = n
            budget -= n

        # 第二优先：从队列里补新请求进来做 prefill
        for req in list(self.waiting):
            if budget <= 0 or len(scheduled) >= self.max_num_seqs:
                break
            n = min(req.num_tokens_to_schedule(), budget)
            scheduled.append(req)
            num_tokens[req.request_id] = n
            budget -= n
            self.waiting.remove(req)
            req.status = "running"
            self.running.append(req)

        self.step_id += 1
        return SchedulerOutput(scheduled, num_tokens)

    def update_from_output(self, sched_out, sampled):
        """对应 vLLM 的 update_from_output：写回采样结果，处理完成与回收。

        本轮被调度但没产生 token 的请求（比如 chunked prefill 的中间块）
        不会出现在 sampled 里，它们保持 running，下一轮继续。
        """
        for req in sched_out.scheduled_reqs:
            if req.request_id not in sampled:
                continue
            req.output_token_ids.append(sampled[req.request_id])
            if req.is_finished:
                req.status = "finished"
                if req in self.running:
                    self.running.remove(req)
                self.finished.append(req)
                req.past = None      # 简化回收；真实 vLLM 走 KVCacheManager.free()


class ModelRunner:
    """对应 vllm/v1/worker/gpu_model_runner.py 的 GPUModelRunner。

    职责：把 Scheduler 排好的一批请求拼成一次前向，返回新采样的 token。

    与真实 vLLM 的差距（要如实知道）：
      · vLLM 用 block_table 让每条序列的 KV 物理上不连续，所以不需要填充；
        这里用「右填充 + 逐序列掩码」对齐，会浪费显存——第 05 章解决。
      · vLLM 会把 prefill 和 decode 混在同一个 batch 里跑；这里分成两组处理，
        纯粹是为了让代码可读，结论不受影响。
      · 输入准备、CUDA graph、attention metadata 这些都被省掉了。
    """

    def __init__(self, model):
        self.model = model

    @torch.no_grad()
    def _run_decode_batch(self, reqs):
        """把一批进度不同的 decode 请求拼成一次前向。"""
        B = len(reqs)
        lens = [r.num_computed_tokens for r in reqs]
        Lmax = max(lens)
        n_layer = self.model.cfg.n_layer

        padded = []
        for layer in range(n_layer):
            ks, vs = [], []
            for r in reqs:
                k, v = r.past[layer]
                pad = Lmax - k.size(2)
                if pad:
                    k = F.pad(k, (0, 0, 0, pad))
                    v = F.pad(v, (0, 0, 0, pad))
                ks.append(k)
                vs.append(v)
            padded.append((torch.cat(ks, 0), torch.cat(vs, 0)))

        # 逐序列掩码：真实历史 [0, L_i) + 新 token 落在下标 Lmax
        S = Lmax + 1
        mask = torch.zeros(B, 1, 1, S, dtype=torch.bool, device=DEVICE)
        for i, r in enumerate(reqs):
            mask[i, 0, 0, : lens[i]] = True
            mask[i, 0, 0, Lmax] = True

        ids = torch.tensor([[r.all_token_ids()[r.num_computed_tokens]] for r in reqs],
                           device=DEVICE)
        pos = torch.tensor(lens, device=DEVICE)
        logits, past = self.model(ids, past_kvs=padded, pos_offset=pos, attn_mask=mask)

        sampled = {}
        for i, r in enumerate(reqs):
            rebuilt = []
            for layer in range(n_layer):
                k_all, v_all = past[layer]
                k = torch.cat([k_all[i:i + 1, :, : lens[i]],
                               k_all[i:i + 1, :, Lmax:Lmax + 1]], dim=2)
                v = torch.cat([v_all[i:i + 1, :, : lens[i]],
                               v_all[i:i + 1, :, Lmax:Lmax + 1]], dim=2)
                rebuilt.append((k, v))
            r.past = rebuilt
            r.num_computed_tokens += 1
            sampled[r.request_id] = int(logits[i, -1].argmax(-1).item())
        return sampled

    @torch.no_grad()
    def execute_model(self, sched_out):
        decode_reqs, prefill_reqs = [], []
        for r in sched_out.scheduled_reqs:
            # 判断依据是「prompt 算完了没有」，而不是「本轮排了几个 token」
            if r.num_computed_tokens >= r.num_prompt_tokens:
                decode_reqs.append(r)
            else:
                prefill_reqs.append(r)

        sampled = {}
        if decode_reqs:
            sampled.update(self._run_decode_batch(decode_reqs))

        for r in prefill_reqs:
            n = sched_out.num_scheduled_tokens[r.request_id]
            start = r.num_computed_tokens
            chunk = r.all_token_ids()[start:start + n]
            toks = torch.tensor([chunk], device=DEVICE)
            logits, past = self.model(toks, past_kvs=r.past, pos_offset=start)
            r.past = past
            r.num_computed_tokens += len(chunk)
            # 只有 prompt 全部算完，才能采样第一个输出 token
            if r.num_computed_tokens >= r.num_prompt_tokens:
                sampled[r.request_id] = int(logits[:, -1].argmax(-1).item())
        return sampled


class EngineCore:
    """对应 vllm/v1/engine/core.py 的 EngineCore。

    整个 vLLM 的推理服务就跑在这三步上：

        schedule()            决定这一轮跑什么
        execute_model()       跑模型
        update_from_output()  把结果写回请求状态

    读懂这个循环你就抓住了 vLLM 的主干。后面所有优化——chunked prefill、
    前缀缓存、抢占、投机解码——都是在这三步里插桩。
    """

    def __init__(self, model, scheduler=None):
        self.scheduler = scheduler or Scheduler()
        self.runner = ModelRunner(model)
        self.step_id = 0
        self.steps = 0

    def step(self):
        sched_out = self.scheduler.schedule()
        if len(sched_out) == 0:
            return None
        sampled = self.runner.execute_model(sched_out)
        self.scheduler.update_from_output(sched_out, sampled)
        self.step_id += 1
        self.steps += 1
        return sampled

    def run(self, max_steps=10000):
        while self.scheduler.has_unfinished() and self.steps < max_steps:
            self.step()
        return self.steps


print(f"引导单元加载完成 | device={DEVICE} dtype={DTYPE} torch={torch.__version__}")
# ===== 引导单元结束 =====

In [ ]:
import time
from collections import Counter, OrderedDict
from statistics import mean, pstdev

model = build_model(block_size=4096)
PREFIX_LEN, SUFFIX_LEN = 256, 32
print(f"参数量 {model.n_params / 1e6:.1f}M，前缀 {PREFIX_LEN} token，后缀 {SUFFIX_LEN} token")

## 一、副本：带前缀缓存的推理实例

每个副本有自己的显存预算（以能缓存的 token 数计）和一套 LRU 前缀缓存。

> 这里的 `Replica` 相当于把第 04、05 章的 `EngineCore` + `KVCacheManager` 打包成一个可独立服务的实例。真实系统里每个副本就是一个独立的 vLLM 进程，本章用同一个模型对象模拟多个副本，是为了让单卡也能跑。

In [ ]:
class Replica:
    def __init__(self, rid, model, capacity_tokens):
        self.rid = rid
        self.model = model
        self.capacity = capacity_tokens
        self.cache = OrderedDict()     # prefix_key -> (past, n_tokens)，按 LRU 淘汰
        self.used = 0
        self.hits = 0
        self.misses = 0
        self.computed_tokens = 0       # 实际算过的 token 数 —— 这就是 TTFT 的来源
        self.load = 0.0                # 衰减的负载计分

    def lookup(self, key):
        if key in self.cache:
            self.cache.move_to_end(key)
            return self.cache[key]
        return None

    def insert(self, key, past, n_tokens):
        if key in self.cache:
            return
        self.cache[key] = (past, n_tokens)
        self.used += n_tokens
        while self.used > self.capacity and len(self.cache) > 1:
            _, (_, n) = self.cache.popitem(last=False)
            self.used -= n

    @torch.no_grad()
    def serve(self, prefix_ids, suffix_ids, key):
        """处理一条请求。命中缓存就只算后缀——这正是 TTFT 的差别所在。"""
        self.load = self.load * 0.9 + (PREFIX_LEN + SUFFIX_LEN)
        hit = self.lookup(key)

        if hit is not None:
            past, plen = hit
            self.model(suffix_ids, past_kvs=past, pos_offset=plen)
            self.computed_tokens += SUFFIX_LEN
            self.hits += 1
            return

        full = torch.cat([prefix_ids, suffix_ids], dim=1)
        _, past = self.model(full)
        self.computed_tokens += full.size(1)
        self.misses += 1
        # 因果注意力保证：前缀部分的 KV 不受后缀影响，可以直接切片出来缓存
        prefix_past = [(k[:, :, :PREFIX_LEN], v[:, :, :PREFIX_LEN]) for k, v in past]
        self.insert(key, prefix_past, PREFIX_LEN)


replicas = [Replica(i, model, capacity_tokens=2048) for i in range(3)]
print(f"启动 {len(replicas)} 个副本，每个前缀缓存容量 2048 token（约 8 条前缀）")

## 二、构造工作负载：热点模板

真实线上流量从不均匀：少数几个 prompt 模板（系统提示、业务指令）占据绝大多数请求。这里用 Zipf 分布模拟。

In [ ]:
def make_workload(n_requests=64, n_templates=8, seed=0):
    g = torch.Generator().manual_seed(seed)
    templates = {
        t: torch.randint(0, model.cfg.vocab_size, (1, PREFIX_LEN), generator=g).to(DEVICE)
        for t in range(n_templates)
    }
    weights = torch.tensor([1.0 / (t + 1) for t in range(n_templates)])
    weights = weights / weights.sum()

    reqs = []
    for _ in range(n_requests):
        t = int(torch.multinomial(weights, 1, generator=g).item())
        suffix = torch.randint(0, model.cfg.vocab_size, (1, SUFFIX_LEN), generator=g).to(DEVICE)
        reqs.append({"key": t, "prefix": templates[t], "suffix": suffix})
    return reqs, templates


workload, templates = make_workload()
dist = Counter(r["key"] for r in workload)
print(f"共 {len(workload)} 条请求，模板分布：{dict(sorted(dist.items()))}")
print()
print("可以看到明显的长尾：模板 0 的请求数远多于模板 7。")

## 三、四种路由策略

| 策略 | 思路 | 预期 |
|---|---|---|
| 轮询 | 挨个分发 | 命中率最低，但绝对均衡 |
| 最小负载 | 谁闲给谁 | 均衡好，但前缀被打散 |
| 前缀亲和 | 同一前缀固定落点 | 命中率最高，但负载倾斜 |
| 混合 | 亲和优先，过载则退避 | 折中 |

In [ ]:
def route_round_robin(replicas, req, state):
    rep = replicas[state["i"] % len(replicas)]
    state["i"] += 1
    return rep


def route_least_load(replicas, req, state):
    return min(replicas, key=lambda r: r.load)


def route_prefix_affinity(replicas, req, state):
    """同一前缀永远落到同一个副本 → 命中率最大化，但可能倾斜。"""
    return replicas[req["key"] % len(replicas)]


def route_hybrid(replicas, req, state, threshold=1.6):
    """先看亲和副本；若它已过载（负载超过均值 threshold 倍），退而选最闲的。"""
    affine = replicas[req["key"] % len(replicas)]
    avg = mean(r.load for r in replicas) + 1e-6
    if affine.load <= avg * threshold:
        return affine
    return min(replicas, key=lambda r: r.load)


STRATEGIES = {
    "轮询": route_round_robin,
    "最小负载": route_least_load,
    "前缀亲和": route_prefix_affinity,
    "混合(亲和+负载)": route_hybrid,
}

## 四、跑实验

In [ ]:
def run(strategy_fn, workload):
    for r in replicas:
        r.cache.clear()
        r.used = r.hits = r.misses = r.computed_tokens = 0
        r.load = 0.0

    state = {"i": 0}
    t0 = time.perf_counter()
    for req in workload:
        rep = strategy_fn(replicas, req, state)
        rep.serve(req["prefix"], req["suffix"], req["key"])
    dt = time.perf_counter() - t0

    total = sum(r.hits + r.misses for r in replicas)
    hits = sum(r.hits for r in replicas)
    computed = sum(r.computed_tokens for r in replicas)
    loads = [r.computed_tokens for r in replicas]
    return {
        "耗时(s)": round(dt, 2),
        "命中率": f"{hits / total:.1%}",
        "总计算token": computed,
        "负载标准差": round(pstdev(loads)),
    }


ideal = len(workload) * SUFFIX_LEN + len(templates) * PREFIX_LEN
print(f"理论上限（每个模板只算一次前缀）: {ideal:,} token")

rows = {name: run(fn, workload) for name, fn in STRATEGIES.items()}

print()
print(f"{'策略':<18}{'命中率':>9}{'总计算token':>14}{'耗时(s)':>10}{'负载标准差':>12}")
print("-" * 64)
for name, r in rows.items():
    print(f"{name:<18}{r['命中率']:>9}{r['总计算token']:>14,}{r['耗时(s)']:>10.2f}{r['负载标准差']:>12,}")

print(f"\n理论上限是 {ideal:,} token，越接近说明缓存利用越充分。")

## 五、结果解读

你应该会看到这样一组关系：

| 策略 | 命中率 | 特点 |
|---|---|---|
| **轮询** | 最低 | 同一前缀被轮流送到所有副本，每个副本都要冷启动 |
| **最小负载** | 也低 | 只看负载不看前缀，热点前缀依然被打散 |
| **前缀亲和** | 最高 | 同前缀固定落点，命中率拉满，**但负载倾斜** |
| **混合** | 接近亲和 | 在亲和与均衡之间取折中 |

这里藏着本项目的**核心权衡**，也是面试要讲的重点：

> **缓存命中率要求"同前缀固定落点"，负载均衡要求"打散落点"，两者天然冲突。**

纯前缀亲和会把热点前缀的全部流量压到一个副本上，那个副本先过载；纯负载均衡则让缓存形同虚设。真实系统必须做折中。

你可以做一个更有说服力的分析：**扫描 `threshold` 参数**，画出"命中率"和"负载标准差"两条曲线，找出拐点。这条曲线就是你的项目结论。

## 六、这个项目还能往哪走

想把它做成真正有分量的作品，下面每一条都可以继续做：

In [ ]:
NEXT_STEPS = """
1. 副本故障与扩缩容
   副本挂了或扩容时，路由表怎么迁移？迁移导致缓存冷启动怎么预热？
   这是纯前缀亲和方案最脆弱的地方。

2. 热点前缀的复制策略
   当一个前缀热到单副本扛不住时，主动把它"灌"到多个副本上
   （每个副本都缓存一份），再对这组副本做负载均衡。
   这就把"固定落点"变成了"固定落点集合"。

3. 与前缀长度的联动
   前缀越长，缓存收益越大，越值得为它牺牲负载均衡；
   短前缀收益小，直接轮询即可。按前缀长度动态决定路由激进程度。

4. 真实框架对接
   这套逻辑在 K8s 生态里对应 Gateway API Inference Extension
   和 llm-d 的 prefix-aware routing；vLLM production stack 也有类似组件。
   把自研路由器换成它们的实现，对比效果和复杂度。

5. 观测指标
   路由层必须暴露：命中率、各副本负载分布、TTFT p50/p99、
   路由表大小、热点前缀 TOP-N。没有这些指标，线上出问题查不出来。
"""
print(NEXT_STEPS)

## 七、写进简历的版本

这个项目的价值不只是"我做了个路由器"，而是它证明你能**从机制推导优化、再用数据验证**：

> 针对多副本推理服务中前缀缓存命中率被负载均衡稀释的问题，设计并实现前缀感知路由层：通过 block 级前缀哈希做落点亲和，叠加负载感知退避避免热点倾斜，在前缀重复度 __% 的流量下将缓存命中率从 __% 提升至 __%，prefill 计算量下降 __%，等效 TTFT p99 下降 __%；并量化了"命中率与负载均衡的冲突边界"，给出按前缀长度动态调整路由激进程度的策略。

注意最后半句：**承认并量化权衡**，比只报一个漂亮百分比可信得多，也是资深工程师的表达方式。

> 如果只在模拟环境验证，就老实写"基于模拟流量"。千万别写成线上数据——面试官一定会追问线上流量分布、灰度方案和回滚策略，答不上来就全盘可疑。

## 八、课程结束，接下来做什么

十个 notebook 走完，你现在应该能够：

- 白板画出一次推理迭代的完整数据流，并指出每一步的瓶颈在哪
- 当场推算给定模型和卡型的最大并发
- 解释 continuous batching / PagedAttention / prefix caching / chunked prefill / 投机解码各自的收益来源**和代价**
- 拿到一个 vLLM 部署，按方法论定位吞吐问题
- 识别不合理的 prompt 结构，并估算改动能省多少钱

**下一步建议**：

1. 把线上真实的一段 prompt 和流量分布脱敏后，套用第 09 章的审计方法跑一遍。
2. 挑一个真正关心的优化点，把这里的模拟换成真实 vLLM 副本，产出带线上口径的数字。
3. 去给 vLLM 或 SGLang 提一个 PR。哪怕只是文档修正，走完一次完整的开源协作流程本身就有价值。

最后提醒一句：这个仓库里所有数字都是在你自己的机器上跑出来的。**面试时不要背这里的数字，要学会这里的推导方式**——推导方式才是别人拿不走的东西。